# Laptop Price Prediction — Data Cleaning

## Objective

The objective of this notebook is to clean the dataset based on the issues identified during exploratory data analysis.

The cleaning process focuses on:

- Removing irrelevant columns
- Validating numerical values
- Checking categorical consistency
- Handling identified data-quality issues
- Saving the cleaned dataset for feature engineering

In [42]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
%matplotlib inline

In [43]:
df = pd.read_csv("../data/raw/laptops.csv")

df.head()

,index,brand,Model,Price,Rating,processor_brand,processor_tier,num_cores,num_threads,ram_memory,...,secondary_storage_type,secondary_storage_capacity,gpu_brand,gpu_type,is_touch_screen,display_size,resolution_width,resolution_height,OS,year_of_warranty
0,1,tecno,Tecno Megabook T1 Laptop (11th Gen Core i3/ 8G...,23990,63,intel,core i3,2,4,8,...,No secondary storage,0,intel,integrated,False,15.6,1920,1080,windows,1
1,2,tecno,Tecno Megabook T1 Laptop (11th Gen Core i7/ 16...,35990,67,intel,core i7,4,8,16,...,No secondary storage,0,intel,integrated,False,15.6,1920,1080,windows,1
2,3,hp,HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen...,51100,73,amd,ryzen 5,6,12,8,...,No secondary storage,0,amd,dedicated,False,15.6,1920,1080,windows,1
3,4,acer,Acer Extensa EX214-53 Laptop (12th Gen Core i5...,39990,62,intel,core i5,12,16,8,...,No secondary storage,0,intel,integrated,False,14.0,1920,1080,windows,1
4,5,lenovo,Lenovo V15 82KDA01BIH Laptop (AMD Ryzen 3 5300...,28580,62,amd,ryzen 3,4,8,8,...,No secondary storage,0,amd,integrated,False,15.6,1920,1080,windows,1


In [44]:
df = df.drop(columns=["index", "Model"])

# "Model" has 991 unique values (essentially one per row) and provides no
# generalizable signal for price prediction, so it is dropped here. The other
# engineered features (brand, processor_tier, ram_memory, etc.) already
# capture the specs that would otherwise need to be parsed out of Model.

## 1. Numerical Value Validation

We validate numerical features to identify physically impossible or invalid values.

Unusually high values are not automatically removed because premium laptops can legitimately have high specifications and prices.

In [45]:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Negative values per column:")
print((df[numeric_columns] < 0).sum())

print("\nZero values per column:")
print((df[numeric_columns] == 0).sum())

Negative values per column:
Price                         0
Rating                        0
num_cores                     0
num_threads                   0
ram_memory                    0
primary_storage_capacity      0
secondary_storage_capacity    0
display_size                  0
resolution_width              0
resolution_height             0
dtype: int64

Zero values per column:
Price                           0
Rating                          0
num_cores                       0
num_threads                     5
ram_memory                      0
primary_storage_capacity        0
secondary_storage_capacity    976
display_size                    0
resolution_width                0
resolution_height               0
dtype: int64


## 2. Categorical Consistency

We inspect categorical values for inconsistent formatting such as unnecessary whitespace or inconsistent capitalization.

In [46]:
categorical_columns = df.select_dtypes(include="str").columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].unique())


brand
<ArrowStringArray>
[    'tecno',        'hp',      'acer',    'lenovo',     'apple',   'infinix',
      'asus',      'dell',   'samsung',       'msi',     'wings',   'ultimus',
 'primebook',     'iball', 'zebronics',     'chuwi',  'gigabyte',       'jio',
     'honor',    'realme',     'avita', 'microsoft',   'fujitsu',        'lg',
    'walker',       'axl']
Length: 26, dtype: str

processor_brand
<ArrowStringArray>
['intel', 'amd', 'apple', 'other']
Length: 4, dtype: str

processor_tier
<ArrowStringArray>
[     'core i3',      'core i7',      'ryzen 5',      'core i5',
      'ryzen 3',           'm1',      'core i9',      'ryzen 7',
        'other',           'm3',           'm2',      'ryzen 9',
      'celeron', 'core ultra 7',      'pentium']
Length: 15, dtype: str

primary_storage_type
<ArrowStringArray>
['SSD', 'HDD']
Length: 2, dtype: str

secondary_storage_type
<ArrowStringArray>
['No secondary storage', 'SSD']
Length: 2, dtype: str

gpu_brand
<ArrowStringArray>
['intel'

## 3. Fixing `year_of_warranty`

`year_of_warranty` mixes numeric strings (`'1'`, `'2'`, `'3'`) with a text placeholder (`'No information'`), so it can't be used as a number as-is. We convert it to a numeric column and let `'No information'` become a proper missing value (`NaN`), which we can decide how to impute or encode in the feature engineering stage.

In [47]:
df['year_of_warranty'] = pd.to_numeric(
    df['year_of_warranty'].replace('No information', np.nan)
)

df['year_of_warranty'].value_counts(dropna=False)

year_of_warranty
1.0    900
2.0     63
NaN     18
3.0     10
Name: count, dtype: int64

## 4. Final Validation

The dataset is now cleaned based on the issues identified during EDA.

We perform a final validation to confirm the dataset has the expected structure. Note: `year_of_warranty` will now show some missing values — this is expected, since we intentionally converted `'No information'` to `NaN` in the previous step, and will be addressed during feature engineering.

In [48]:
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values by column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Dataset shape: (991, 20)
Duplicate rows: 7

Missing values by column:
year_of_warranty    18
dtype: int64


## 5. Save Cleaned Dataset

The cleaned dataset is saved separately from the raw dataset so that the original data remains unchanged and the cleaning process can be reproduced.

In [50]:
df.to_csv("../data/processed/laptops_cleaned.csv", index=False)

## Conclusion

The raw laptop dataset was cleaned by removing irrelevant columns, standardizing categorical values, converting unavailable entries to missing values, and validating numerical features.

The cleaned dataset was saved as `laptops_cleaned.csv` and is used as the input for feature engineering.